# Coimbatore Urban Intelligence — Exploratory Data Analysis (EDA)

This notebook performs preprocessing, descriptive statistical analysis, and basic visualizations for the Coimbatore Urban Intelligence platform datasets:
1. Air Quality (`01_air_quality.csv`)
2. Weather (`02_weather.csv`)
3. Traffic (`03_traffic.csv`)
4. Civic Issues (`04_civic_issues.csv`)
5. Area Master (`05_area_master.csv`)
6. Commercial Activity (`06_commercial_activity.csv`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Connect to SQLite DB
conn = sqlite3.connect('../coimbatore_urban.db')
print("Connected to database successfully!")

## 1. Load Datasets & Check Shapes

In [ ]:
df_area = pd.read_sql_query("SELECT * FROM area_master", conn)
df_weather = pd.read_sql_query("SELECT * FROM weather", conn)
df_traffic = pd.read_sql_query("SELECT * FROM traffic", conn)
df_aq = pd.read_sql_query("SELECT * FROM air_quality", conn)
df_civic = pd.read_sql_query("SELECT * FROM civic_issues", conn)
df_comm = pd.read_sql_query("SELECT * FROM commercial_activity", conn)

print("Area Master shape:", df_area.shape)
print("Weather shape:", df_weather.shape)
print("Traffic shape:", df_traffic.shape)
print("Air Quality shape:", df_aq.shape)
print("Civic Issues shape:", df_civic.shape)
print("Commercial Activity shape:", df_comm.shape)

## 2. Preprocessing & Data Aggregation

In [ ]:
# Preprocess Traffic: convert date to datetime and calculate daily average
df_traffic['date'] = pd.to_datetime(df_traffic['date'])
daily_traffic = df_traffic.groupby(['date', 'area'])['vehicle_count'].mean().reset_index()

# Preprocess AQI: Calculate average PM2.5
df_aq['date'] = pd.to_datetime(df_aq['date'])
df_weather['date'] = pd.to_datetime(df_weather['date'])

# Merge datasets for Cross-Domain analysis
df_merged = pd.merge(df_aq, daily_traffic, on=['date', 'area'])
df_merged = pd.merge(df_merged, df_weather, on='date')
df_merged = pd.merge(df_merged, df_area, on='area')

print("Merged Dataset for Cross-Domain Analysis:")
df_merged.head()

## 3. Correlation Analysis (Traffic, AQI & Weather)

In [ ]:
correlation_cols = ['pm25', 'pm10', 'no2', 'so2', 'vehicle_count', 'temperature', 'humidity', 'rainfall', 'wind_speed']
corr_matrix = df_merged[correlation_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Coimbatore Cross-Domain Correlation Matrix')
plt.show()

## 4. Key EDA Visualizations
### Traffic Density by Area

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_traffic, x='area', y='vehicle_count')
plt.xticks(rotation=45)
plt.title('Vehicle Count Distribution by Coimbatore Area')
plt.ylabel('Vehicle Count')
plt.xlabel('Area')
plt.show()

### AQI vs Vehicle Count

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_merged, x='vehicle_count', y='pm25', hue='area_type', alpha=0.8)
plt.title('PM2.5 Pollution vs Average Daily Vehicle Count')
plt.xlabel('Average Vehicle Count')
plt.ylabel('PM2.5 Concentration')
plt.legend(title='Area Type')
plt.show()

### Civic Complaints Resolution Time

In [ ]:
# Filter out unresolved issues for resolution analysis
df_resolved = df_civic[df_civic['status'] == 'Resolved'].copy()
df_resolved['resolution_days'] = pd.to_numeric(df_resolved['resolution_days'])

plt.figure(figsize=(10, 6))
sns.barplot(data=df_resolved, x='issue_type', y='resolution_days', hue='severity', errorbar=None)
plt.title('Average Resolution Days by Complaint Type and Severity')
plt.xticks(rotation=30)
plt.ylabel('Resolution Days')
plt.xlabel('Complaint Category')
plt.show()

## 5. Close Connection

In [ ]:
conn.close()
print("Database connection closed. EDA finished.")